# N2 · arxiv 分诊: 把洪流变成本周阅读清单 (Triage)

> 配套 9.1-L4 · **真实科研动作**: 用你的研究关键词给一批论文打分排序, 自动产出「本周精读清单」。
>
> 治的病: arxiv 每天上百篇, 焦虑性囤积。解药: 把「读不读」从情绪决策变成参数决策。

In [1]:
import sys
from pathlib import Path
import pandas as pd

SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))
import arxiv_triage
print("已载入 arxiv_triage, 内置样例 feed 有", len(arxiv_triage.SAMPLE_FEED), "篇")

已载入 arxiv_triage, 内置样例 feed 有 6 篇


## 1. 定义你的研究关键词权重

这是整个分诊的**唯一参数**: 你最在意什么, 给它高权重。写在一个地方, 随方向演进去调它,
而不是每天重新纠结读不读。下面用一个「做 reasoning RL + 关心可复现」的博0画像举例。

In [2]:
# 你最在意的方向给高权重 (这就是把"读不读"参数化)
weights = {
    "reasoning": 3,
    "rl": 2,
    "dpo": 2,
    "alignment": 2,
    "reproducibility": 3,
    "scaling": 2,
}
weights

{'reasoning': 3,
 'rl': 2,
 'dpo': 2,
 'alignment': 2,
 'reproducibility': 3,
 'scaling': 2}

## 2. 全量打分 (可解释: 每篇命中了哪些关键词)

In [3]:
ranked = arxiv_triage.triage(arxiv_triage.SAMPLE_FEED, weights)
df = pd.DataFrame([{
    "score": p["score"],
    "title": p["title"][:52],
    "命中关键词": ", ".join(p["hits"]) or "—",
} for p in ranked])
df

,score,title,命中关键词
0,17.0,Scaling Laws for Reasoning RL,"reasoning, rl, scaling"
1,17.0,On the Reproducibility of RLHF Pipelines,"reasoning, rl, alignment, reproducibility"
2,12.0,Process Reward Models for Multi-step Reasoning,reasoning
3,8.0,Robust DPO under Noisy Preference Labels,"dpo, alignment"
4,0.0,A Survey of Vision Transformers for Medical Im...,—
5,0.0,Efficient Quantization for Edge Deployment of ...,—


## 3. 本周精读清单 (只取分数 > 0 的前几篇, 0 分的直接归档)

注意那两篇 **医学 ViT** 和 **CNN 量化** 的分数应该是 0 —— 它们和你的方向无关,
分诊系统自动把它们挡在精读队列外。**这就是系统替你做了「不读」的决定, 省下你的注意力。**

In [4]:
top = arxiv_triage.reading_list(arxiv_triage.SAMPLE_FEED, weights, top_k=4)
print("📖 本周精读清单 (按相关分排序):\n")
for i, p in enumerate(top, 1):
    print(f"{i}. [{p['score']:>5}] {p['title']}")
    print(f"         命中: {', '.join(p['hits'])}\n")

📖 本周精读清单 (按相关分排序):

1. [ 17.0] Scaling Laws for Reasoning RL
         命中: reasoning, rl, scaling

2. [ 17.0] On the Reproducibility of RLHF Pipelines
         命中: reasoning, rl, alignment, reproducibility

3. [ 12.0] Process Reward Models for Multi-step Reasoning
         命中: reasoning

4. [  8.0] Robust DPO under Noisy Preference Labels
         命中: dpo, alignment



## 4. (可选) 连真实 arxiv

`fetch_live` 会尝试连 arxiv 公开 API; 离线/超时会自动兜底回内置样例, 所以这一格在任何环境都不报错。
真用时把 query 换成你的方向, 例如 `cat:cs.CL AND abs:reasoning`。

In [5]:
live = arxiv_triage.fetch_live("cat:cs.CL", max_results=5)
print(f"取到 {len(live)} 篇 (可能是真实 arxiv, 也可能是离线兜底样例)")
for p in arxiv_triage.reading_list(live, weights, top_k=3):
    print(f"  [{p['score']:>5}] {p['title'][:60]}")

[arxiv_triage] live fetch 失败 (<HTTPError 502: 'Bad Gateway'>), 改用内置样例 feed.
取到 6 篇 (可能是真实 arxiv, 也可能是离线兜底样例)
  [ 17.0] Scaling Laws for Reasoning RL
  [ 17.0] On the Reproducibility of RLHF Pipelines
  [ 12.0] Process Reward Models for Multi-step Reasoning


## 5. 反思

你刚把一批论文从「一堆标题」变成了「一个排好序、带理由的队列」。把它接进 9.1-L4 的 weekly review:

- **每周** 跑一次这个分诊 → 得到 3-5 篇精读清单。
- 精读用 **9.3 三遍读法** → 产出文献卡 (N1) + 永久笔记 (L2)。
- 读出的开放问题 → 扔进 **idea inbox** (L3)。

**整个 Module 9 是一条流水线, 你刚跑通了它的「进料」这一段。**

> 现实建议: 别追求关键词权重「完美」。先用一个粗糙版本跑起来, 在每周 review 里微调。
> 一个在用的粗系统, 远胜一个完美但没启动的系统。